# ResGRU-UNet PM2.5 - model card

Loads the [ResGRU-UNet PM2.5 forecasting model](https://www.kaggle.com/models/imvetri0/resgru-unet-pm25) without the challenge data: prints its card, lists the files, loads the five networks and runs one forward pass on a random input.

- Five ResGRU-UNet networks (seeds 42, 1337, 1, 2 and 3), 7,122,113 parameters each, that forecast PM2.5 for the next 16 hours at every cell of a 140 x 124 grid over India.
- The five-seed mean with the shipped factors scored 0.8914 on the private test set and 0.9056 on the public one, after the deadline.
- Paper and code: https://github.com/Vetri-78640/pm2.5-backbone-study-pipeline-over-backbone

In [ ]:
import os, sys
import numpy as np
import torch

MODEL = next(dp for dp, _, fs in os.walk('/kaggle/input') if 'predict.py' in fs)
print('model folder:', MODEL)

## 1. Model card

In [ ]:
print(open(os.path.join(MODEL, 'README.md')).read())

## 2. Files

In [ ]:
for f in sorted(os.listdir(MODEL)):
    print(f'{os.path.getsize(os.path.join(MODEL, f)):>12,}  {f}')

## 3. The five networks

`study.py` is imported from the model folder, as `predict.py` does. Importing it only defines the pipeline and network code: no pipeline step runs and no data is read.

In [ ]:
sys.path.insert(0, MODEL)
import study as S

for s in [42, 1337, 1, 2, 3]:
    ck = torch.load(os.path.join(MODEL, f'resgru_s{s}_phaseA_best.pt'), map_location='cpu', weights_only=False)
    model = S.ResGRUUNet(S.C.C_IN)
    model.load_state_dict(ck['model'], strict=True)
    n = sum(p.numel() for p in model.parameters())
    print(f'seed {s:>4}: {n:,} parameters, best epoch {ck["epoch"]}, logged validation score {ck["metrics"]["score"]:.5f}')

## 4. Calibration factors, one per lead hour

In [ ]:
hs = np.load(os.path.join(MODEL, 'horizon_scales.npy'))
for h, s in enumerate(hs, 1):
    print(f'lead hour {h:2d}: {s:.2f}')

## 5. One forward pass on a random input

The seed-3 network, the last one loaded above, on a random window: 20 channels x 10 hours on the 140 x 124 grid, plus the last observed log(1 + PM2.5) frame. The output is log(1 + PM2.5) for the 16 lead hours.

In [ ]:
torch.manual_seed(0)
x = torch.randn(1, 20, 10, 140, 124)
last = torch.randn(1, 140, 124)
model.eval()
with torch.no_grad():
    y = model(x, last)
assert tuple(y.shape) == (1, 16, 140, 124)
print('input', tuple(x.shape), '+ last frame', tuple(last.shape), '-> output', tuple(y.shape))

## 6. Forecast for the test set

`predict.py` builds the 218 test windows, averages the five networks, multiplies lead hour h by its factor and writes `preds.npy` with shape (218, 140, 124, 16): window, row, column, lead hour. `--no-cal` skips the factors.

It needs the challenge data attached as well as this model. Kaggle provides the data to the challenge's participants, so it is not run here. A check run with the data gave output identical to the submitted forecast, with and without the factors.

In [ ]:
print(f'python {MODEL}/predict.py            # five-seed mean with the factors')
print(f'python {MODEL}/predict.py --no-cal   # five-seed mean without the factors')